# Spring Boot Learning Guide

This notebook covers the fundamentals of Spring Boot development, including application setup, REST endpoints, and dependency injection.

**Spring Boot Overview:**
- Framework for building production-grade Spring applications
- Simplified configuration and rapid application development
- Built-in servers (Tomcat, Jetty)
- Starter dependencies for easy integration

## 1. Project Setup and Dependencies

To create a Spring Boot project, you need:
- **Maven** or **Gradle** as build tool
- **Java Development Kit (JDK)** 8 or higher
- **Spring Boot Starter Web** for building web applications

**pom.xml dependencies:**
```xml
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-web</artifactId>
</dependency>
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-data-jpa</artifactId>
</dependency>
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-validation</artifactId>
</dependency>
```

In [ ]:
// Example: Project structure
/*
my-spring-boot-app/
├── src/
│   ├── main/
│   │   ├── java/
│   │   │   └── com/example/
│   │   │       ├── SpreadsheetApp.java (Main class)
│   │   │       ├── controller/
│   │   │       ├── service/
│   │   │       ├── repository/
│   │   │       └── model/
│   │   └── resources/
│   │       ├── application.properties
│   │       └── application.yml
│   └── test/
│       └── java/
├── pom.xml
└── README.md
*/

## 2. Create Simple Spring Boot Application

The main application class initializes the Spring Boot application with the `@SpringBootApplication` annotation.

In [ ]:
package com.example;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class SpreadsheetApp {
    
    public static void main(String[] args) {
        SpringApplication.run(SpreadsheetApp.class, args);
    }
}

// @SpringBootApplication combines:
// - @Configuration: Defines this as a configuration class
// - @EnableAutoConfiguration: Enables auto-configuration
// - @ComponentScan: Scans for Spring components in the current package

## 3. Configure Application Properties

Configuration can be done via `application.properties` or `application.yml` files.

**application.properties:**

In [ ]:
# Application Properties
spring.application.name=Spreadsheet-App
server.port=8080
spring.jpa.hibernate.ddl-auto=update
spring.datasource.url=jdbc:mysql://localhost:3306/spreadsheet_db
spring.datasource.username=root
spring.datasource.password=password
spring.datasource.driver-class-name=com.mysql.cj.jdbc.Driver
spring.jpa.show-sql=true
spring.jpa.properties.hibernate.format_sql=true

**application.yml alternative:**

In [ ]:
spring:
  application:
    name: Spreadsheet-App
  jpa:
    hibernate:
      ddl-auto: update
    show-sql: true
    properties:
      hibernate:
        format_sql: true
  datasource:
    url: jdbc:mysql://localhost:3306/spreadsheet_db
    username: root
    password: password
    driver-class-name: com.mysql.cj.jdbc.Driver

server:
  port: 8080

## 4. Build REST Endpoints

REST controllers handle HTTP requests and return responses. Use `@RestController` and `@RequestMapping` annotations.

In [ ]:
package com.example.controller;

import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@RestController
@RequestMapping("/api/users")
public class UserController {
    
    @Autowired
    private UserService userService;
    
    // GET: Retrieve all users
    @GetMapping
    public ResponseEntity<List<User>> getAllUsers() {
        List<User> users = userService.getAllUsers();
        return ResponseEntity.ok(users);
    }
    
    // GET: Retrieve user by ID
    @GetMapping("/{id}")
    public ResponseEntity<User> getUserById(@PathVariable Long id) {
        User user = userService.getUserById(id);
        return ResponseEntity.ok(user);
    }
    
    // POST: Create a new user
    @PostMapping
    public ResponseEntity<User> createUser(@RequestBody User user) {
        User savedUser = userService.saveUser(user);
        return ResponseEntity.status(HttpStatus.CREATED).body(savedUser);
    }
    
    // PUT: Update an existing user
    @PutMapping("/{id}")
    public ResponseEntity<User> updateUser(@PathVariable Long id, @RequestBody User user) {
        user.setId(id);
        User updatedUser = userService.saveUser(user);
        return ResponseEntity.ok(updatedUser);
    }
    
    // DELETE: Remove a user
    @DeleteMapping("/{id}")
    public ResponseEntity<String> deleteUser(@PathVariable Long id) {
        userService.deleteUser(id);
        return ResponseEntity.ok("User deleted successfully");
    }
}

## 5. Handle HTTP Requests and Responses

Different annotations for handling request parameters and bodies

In [ ]:
package com.example.controller;

import org.springframework.web.bind.annotation.*;
import java.util.Map;

@RestController
@RequestMapping("/api/request-handling")
public class RequestHandlingController {
    
    // @RequestParam: Query parameters
    // GET /api/request-handling/search?name=John&age=25
    @GetMapping("/search")
    public String searchUser(
            @RequestParam String name,
            @RequestParam(required = false) Integer age) {
        return "Searching for user: " + name + " Age: " + age;
    }
    
    // @PathVariable: Path parameters
    // GET /api/request-handling/users/123
    @GetMapping("/users/{userId}")
    public String getUserInfo(@PathVariable Long userId) {
        return "User ID: " + userId;
    }
    
    // @RequestBody: Request body (JSON)
    // POST /api/request-handling/register
    @PostMapping("/register")
    public String registerUser(@RequestBody User user) {
        return "User registered: " + user.getName();
    }
    
    // @RequestHeader: Access HTTP headers
    @GetMapping("/headers")
    public String getHeaders(@RequestHeader("Authorization") String token) {
        return "Token: " + token;
    }
    
    // Multiple parameters combined
    @PutMapping("/update/{id}")
    public String updateWithBody(
            @PathVariable Long id,
            @RequestBody User user,
            @RequestParam String action) {
        return "Updating user " + id + " with action: " + action;
    }
}

// User model class
class User {
    private Long id;
    private String name;
    private String email;
    
    // Constructors, getters, setters
    public User(String name, String email) {
        this.name = name;
        this.email = email;
    }
    
    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    
    public String getName() { return name; }
    public void setName(String name) { this.name = name; }
    
    public String getEmail() { return email; }
    public void setEmail(String email) { this.email = email; }
}

## 6. Implement Dependency Injection

Spring's IoC container manages objects and their dependencies using `@Autowired` and constructor injection.

In [ ]:
// Service Interface
package com.example.service;

import com.example.model.User;
import java.util.List;

public interface UserService {
    List<User> getAllUsers();
    User getUserById(Long id);
    User saveUser(User user);
    void deleteUser(Long id);
}

// Service Implementation
package com.example.service;

import com.example.model.User;
import com.example.repository.UserRepository;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.stereotype.Service;
import java.util.List;
import java.util.Optional;

@Service
public class UserServiceImpl implements UserService {
    
    // Constructor Injection (Recommended)
    private final UserRepository userRepository;
    
    public UserServiceImpl(UserRepository userRepository) {
        this.userRepository = userRepository;
    }
    
    @Override
    public List<User> getAllUsers() {
        return userRepository.findAll();
    }
    
    @Override
    public User getUserById(Long id) {
        Optional<User> user = userRepository.findById(id);
        return user.orElseThrow(() -> new RuntimeException("User not found"));
    }
    
    @Override
    public User saveUser(User user) {
        return userRepository.save(user);
    }
    
    @Override
    public void deleteUser(Long id) {
        userRepository.deleteById(id);
    }
}

// Alternative: @Autowired Field Injection (Less preferred)
@Service
public class UserServiceAlt implements UserService {
    @Autowired
    private UserRepository userRepository;
    // ... rest of implementation
}

In [ ]:
// Repository
package com.example.repository;

import com.example.model.User;
import org.springframework.data.jpa.repository.JpaRepository;
import org.springframework.stereotype.Repository;

@Repository
public interface UserRepository extends JpaRepository<User, Long> {
    User findByEmail(String email);
    User findByName(String name);
}

// Entity Model
package com.example.model;

import jakarta.persistence.*;

@Entity
@Table(name = "users")
public class User {
    
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    
    @Column(nullable = false)
    private String name;
    
    @Column(unique = true, nullable = false)
    private String email;
    
    private String phone;
    
    public User() {}
    
    public User(String name, String email) {
        this.name = name;
        this.email = email;
    }
    
    // Getters and Setters
    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    
    public String getName() { return name; }
    public void setName(String name) { this.name = name; }
    
    public String getEmail() { return email; }
    public void setEmail(String email) { this.email = email; }
    
    public String getPhone() { return phone; }
    public void setPhone(String phone) { this.phone = phone; }
}

## 7. Add Request Validation

Validate incoming data using `@Valid` and Jakarta Validation annotations.

In [ ]:
package com.example.model;

import jakarta.validation.constraints.*;

public class ValidatedUser {
    
    @NotBlank(message = "Name cannot be blank")
    @Size(min = 2, max = 100, message = "Name must be between 2 and 100 characters")
    private String name;
    
    @NotNull(message = "Email cannot be null")
    @Email(message = "Email should be valid")
    private String email;
    
    @Min(value = 18, message = "Age must be at least 18")
    @Max(value = 120, message = "Age must not exceed 120")
    private Integer age;
    
    @Pattern(regexp = "^[0-9]{10}$", message = "Phone must be 10 digits")
    private String phone;
    
    @NotEmpty(message = "Address cannot be empty")
    private String address;
    
    // Constructors
    public ValidatedUser() {}
    
    public ValidatedUser(String name, String email, Integer age) {
        this.name = name;
        this.email = email;
        this.age = age;
    }
    
    // Getters and Setters
    public String getName() { return name; }
    public void setName(String name) { this.name = name; }
    
    public String getEmail() { return email; }
    public void setEmail(String email) { this.email = email; }
    
    public Integer getAge() { return age; }
    public void setAge(Integer age) { this.age = age; }
    
    public String getPhone() { return phone; }
    public void setPhone(String phone) { this.phone = phone; }
    
    public String getAddress() { return address; }
    public void setAddress(String address) { this.address = address; }
}

In [ ]:
// Controller with Validation
package com.example.controller;

import com.example.model.ValidatedUser;
import jakarta.validation.Valid;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;
import org.springframework.web.bind.MethodArgumentNotValidException;

@RestController
@RequestMapping("/api/validated-users")
public class ValidatedUserController {
    
    @PostMapping
    public ResponseEntity<String> createUser(@Valid @RequestBody ValidatedUser user) {
        return ResponseEntity.status(HttpStatus.CREATED).body("User created successfully");
    }
    
    // Global Exception Handler for validation errors
    @ExceptionHandler(MethodArgumentNotValidException.class)
    public ResponseEntity<Object> handleValidationException(MethodArgumentNotValidException ex) {
        return ResponseEntity.status(HttpStatus.BAD_REQUEST).body(
            "Validation Error: " + ex.getBindingResult().getAllErrors().get(0).getDefaultMessage()
        );
    }
}

// Example validation constraints available:
/*
@NotNull - value must not be null
@NotEmpty - value must not be null or empty
@NotBlank - value must not be null or whitespace
@Min - numeric value must be >= minimum
@Max - numeric value must be <= maximum
@DecimalMin - decimal value must be >= minimum
@DecimalMax - decimal value must be <= maximum
@Size - string length must be between min and max
@Email - value must be valid email format
@Pattern - value must match regex pattern
@Positive - value must be positive
@Negative - value must be negative
@AssertTrue - value must be true
@AssertFalse - value must be false
*/

## 8. Test the Endpoints

Write unit tests using JUnit 5 and MockMvc to test REST endpoints.

## 9. Global Exception Handling

Create centralized exception handling for the entire application using `@RestControllerAdvice`.

In [ ]:
package com.example.exception;

public class ResourceNotFoundException extends RuntimeException {
    public ResourceNotFoundException(String message) {
        super(message);
    }
}

// Error Response DTO
package com.example.dto;

import java.time.LocalDateTime;

public class ErrorResponse {
    private String message;
    private int status;
    private String error;
    private LocalDateTime timestamp;
    private String path;
    
    public ErrorResponse(String message, int status, String error) {
        this.message = message;
        this.status = status;
        this.error = error;
        this.timestamp = LocalDateTime.now();
    }
    
    // Getters and Setters
    public String getMessage() { return message; }
    public void setMessage(String message) { this.message = message; }
    
    public int getStatus() { return status; }
    public void setStatus(int status) { this.status = status; }
    
    public String getError() { return error; }
    public void setError(String error) { this.error = error; }
    
    public LocalDateTime getTimestamp() { return timestamp; }
    public void setTimestamp(LocalDateTime timestamp) { this.timestamp = timestamp; }
    
    public String getPath() { return path; }
    public void setPath(String path) { this.path = path; }
}

// Global Exception Handler
package com.example.exception;

import com.example.dto.ErrorResponse;
import org.springframework.http.HttpStatus;
import org.springframework.http.ResponseEntity;
import org.springframework.validation.FieldError;
import org.springframework.web.bind.MethodArgumentNotValidException;
import org.springframework.web.bind.annotation.ExceptionHandler;
import org.springframework.web.bind.annotation.RestControllerAdvice;
import org.springframework.web.context.request.WebRequest;
import org.springframework.web.servlet.mvc.method.annotation.ResponseEntityExceptionHandler;

import java.util.HashMap;
import java.util.Map;

@RestControllerAdvice
public class GlobalExceptionHandler extends ResponseEntityExceptionHandler {
    
    @ExceptionHandler(ResourceNotFoundException.class)
    public ResponseEntity<ErrorResponse> handleResourceNotFoundException(
            ResourceNotFoundException ex, WebRequest request) {
        ErrorResponse error = new ErrorResponse(
            ex.getMessage(),
            HttpStatus.NOT_FOUND.value(),
            "Resource Not Found"
        );
        error.setPath(request.getDescription(false).replace("uri=", ""));
        return ResponseEntity.status(HttpStatus.NOT_FOUND).body(error);
    }
    
    @ExceptionHandler(MethodArgumentNotValidException.class)
    public ResponseEntity<Map<String, Object>> handleValidationException(
            MethodArgumentNotValidException ex) {
        Map<String, String> errors = new HashMap<>();
        ex.getBindingResult().getAllErrors().forEach(error ->
            errors.put(((FieldError) error).getField(), error.getDefaultMessage())
        );
        
        Map<String, Object> response = new HashMap<>();
        response.put("status", HttpStatus.BAD_REQUEST.value());
        response.put("error", "Validation Failed");
        response.put("message", errors);
        
        return ResponseEntity.status(HttpStatus.BAD_REQUEST).body(response);
    }
    
    @ExceptionHandler(Exception.class)
    public ResponseEntity<ErrorResponse> handleGlobalException(
            Exception ex, WebRequest request) {
        ErrorResponse error = new ErrorResponse(
            "An unexpected error occurred: " + ex.getMessage(),
            HttpStatus.INTERNAL_SERVER_ERROR.value(),
            "Internal Server Error"
        );
        error.setPath(request.getDescription(false).replace("uri=", ""));
        return ResponseEntity.status(HttpStatus.INTERNAL_SERVER_ERROR).body(error);
    }
}

## 10. Logging Best Practices

Implement structured logging using SLF4J and Logback for better debugging and monitoring.

In [ ]:
package com.example.service;

import org.slf4j.Logger;
import org.slf4j.LoggerFactory;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.stereotype.Service;
import com.example.repository.UserRepository;
import com.example.model.User;

@Service
public class UserServiceWithLogging {
    
    private static final Logger logger = LoggerFactory.getLogger(UserServiceWithLogging.class);
    
    @Autowired
    private UserRepository userRepository;
    
    public User createUser(User user) {
        logger.info("Creating new user with email: {}", user.getEmail());
        try {
            User savedUser = userRepository.save(user);
            logger.debug("User saved successfully with ID: {}", savedUser.getId());
            return savedUser;
        } catch (Exception e) {
            logger.error("Error creating user with email: {}", user.getEmail(), e);
            throw e;
        }
    }
    
    public User getUserById(Long id) {
        logger.trace("Fetching user with ID: {}", id);
        return userRepository.findById(id).orElseThrow(() -> {
            logger.warn("User not found with ID: {}", id);
            return new RuntimeException("User not found");
        });
    }
    
    public void deleteUser(Long id) {
        logger.info("Deleting user with ID: {}", id);
        userRepository.deleteById(id);
        logger.debug("User deleted successfully with ID: {}", id);
    }
}

// Logback Configuration (logback-spring.xml)
/*
<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <property name="LOG_FILE" value="logs/app.log"/>
    
    <appender name="CONSOLE" class="ch.qos.logback.core.ConsoleAppender">
        <encoder>
            <pattern>%d{HH:mm:ss.SSS} [%thread] %-5level %logger{36} - %msg%n</pattern>
        </encoder>
    </appender>
    
    <appender name="FILE" class="ch.qos.logback.core.rolling.RollingFileAppender">
        <file>${LOG_FILE}</file>
        <encoder>
            <pattern>%d{yyyy-MM-dd HH:mm:ss} [%thread] %-5level %logger{36} - %msg%n</pattern>
        </encoder>
        <rollingPolicy class="ch.qos.logback.core.rolling.SizeAndTimeBasedRollingPolicy">
            <fileNamePattern>${LOG_FILE}.%d{yyyy-MM-dd}.%i.gz</fileNamePattern>
            <maxFileSize>10MB</maxFileSize>
        </rollingPolicy>
    </appender>
    
    <root level="INFO">
        <appender-ref ref="CONSOLE"/>
        <appender-ref ref="FILE"/>
    </root>
    
    <logger name="com.example" level="DEBUG"/>
</configuration>
*/

## 11. Spring Security - Authentication & Authorization

Implement JWT-based authentication and role-based access control.

In [ ]:
// Security Configuration
package com.example.config;

import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;
import org.springframework.security.authentication.AuthenticationManager;
import org.springframework.security.config.annotation.authentication.configuration.AuthenticationConfiguration;
import org.springframework.security.config.annotation.method.configuration.EnableGlobalMethodSecurity;
import org.springframework.security.config.annotation.web.builders.HttpSecurity;
import org.springframework.security.config.http.SessionCreationPolicy;
import org.springframework.security.crypto.bcrypt.BCryptPasswordEncoder;
import org.springframework.security.crypto.password.PasswordEncoder;
import org.springframework.security.web.SecurityFilterChain;

@Configuration
@EnableGlobalMethodSecurity(prePostEnabled = true)
public class SecurityConfig {
    
    @Bean
    public PasswordEncoder passwordEncoder() {
        return new BCryptPasswordEncoder();
    }
    
    @Bean
    public AuthenticationManager authenticationManager(AuthenticationConfiguration config) throws Exception {
        return config.getAuthenticationManager();
    }
    
    @Bean
    public SecurityFilterChain filterChain(HttpSecurity http) throws Exception {
        http
            .csrf().disable()
            .exceptionHandling()
            .and()
            .sessionManagement()
            .sessionCreationPolicy(SessionCreationPolicy.STATELESS)
            .and()
            .authorizeRequests()
            .antMatchers("/api/auth/**").permitAll()
            .antMatchers("/api/public/**").permitAll()
            .anyRequest().authenticated();
        
        return http.build();
    }
}

// User Model with Security
package com.example.model;

import jakarta.persistence.*;
import java.util.HashSet;
import java.util.Set;

@Entity
@Table(name = "users")
public class SecurityUser {
    
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    
    @Column(unique = true, nullable = false)
    private String username;
    
    @Column(nullable = false)
    private String password;
    
    @Column(unique = true, nullable = false)
    private String email;
    
    @ManyToMany(fetch = FetchType.EAGER)
    @JoinTable(
        name = "user_roles",
        joinColumns = @JoinColumn(name = "user_id"),
        inverseJoinColumns = @JoinColumn(name = "role_id")
    )
    private Set<Role> roles = new HashSet<>();
    
    private boolean enabled = true;
    
    // Getters and Setters
    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    
    public String getUsername() { return username; }
    public void setUsername(String username) { this.username = username; }
    
    public String getPassword() { return password; }
    public void setPassword(String password) { this.password = password; }
    
    public Set<Role> getRoles() { return roles; }
    public void setRoles(Set<Role> roles) { this.roles = roles; }
}

// Role Model
@Entity
@Table(name = "roles")
public class Role {
    
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    
    @Column(unique = true, nullable = false)
    private String name;
    
    public Role() {}
    public Role(String name) { this.name = name; }
    
    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    
    public String getName() { return name; }
    public void setName(String name) { this.name = name; }
}

// Authentication Controller
package com.example.controller;

import com.example.model.SecurityUser;
import com.example.service.UserSecurityService;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.http.ResponseEntity;
import org.springframework.security.access.prepost.PreAuthorize;
import org.springframework.web.bind.annotation.*;

@RestController
@RequestMapping("/api/auth")
public class AuthController {
    
    @Autowired
    private UserSecurityService userSecurityService;
    
    @PostMapping("/register")
    public ResponseEntity<String> registerUser(@RequestBody SecurityUser user) {
        userSecurityService.registerUser(user);
        return ResponseEntity.ok("User registered successfully");
    }
    
    @PostMapping("/login")
    public ResponseEntity<String> loginUser(@RequestBody LoginRequest request) {
        String token = userSecurityService.login(request.getUsername(), request.getPassword());
        return ResponseEntity.ok(token);
    }
    
    @GetMapping("/admin")
    @PreAuthorize("hasRole('ADMIN')")
    public ResponseEntity<String> adminOnly() {
        return ResponseEntity.ok("Admin access granted");
    }
    
    @GetMapping("/user")
    @PreAuthorize("hasRole('USER')")
    public ResponseEntity<String> userAccess() {
        return ResponseEntity.ok("User access granted");
    }
}

// Login Request DTO
class LoginRequest {
    private String username;
    private String password;
    
    public String getUsername() { return username; }
    public void setUsername(String username) { this.username = username; }
    
    public String getPassword() { return password; }
    public void setPassword(String password) { this.password = password; }
}

## 12. Caching & Performance Optimization

Implement caching using Spring Cache abstraction and Redis for high-performance applications.

In [ ]:
// Cache Configuration
package com.example.config;

import org.springframework.cache.CacheManager;
import org.springframework.cache.annotation.EnableCaching;
import org.springframework.cache.concurrent.ConcurrentMapCacheManager;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;

@Configuration
@EnableCaching
public class CacheConfig {
    
    @Bean
    public CacheManager cacheManager() {
        return new ConcurrentMapCacheManager("users", "products", "orders");
    }
}

// Service with Caching
package com.example.service;

import com.example.model.User;
import com.example.repository.UserRepository;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.cache.annotation.Cacheable;
import org.springframework.cache.annotation.CacheEvict;
import org.springframework.cache.annotation.CachePut;
import org.springframework.stereotype.Service;

@Service
public class UserServiceWithCache {
    
    @Autowired
    private UserRepository userRepository;
    
    // Retrieve from cache if exists, otherwise fetch from DB
    @Cacheable(value = "users", key = "#id")
    public User getUserById(Long id) {
        System.out.println("Fetching user from database: " + id);
        return userRepository.findById(id).orElseThrow();
    }
    
    // Update cache with new value
    @CachePut(value = "users", key = "#user.id")
    public User updateUser(User user) {
        return userRepository.save(user);
    }
    
    // Create user and add to cache
    @CachePut(value = "users", key = "#result.id")
    public User createUser(User user) {
        return userRepository.save(user);
    }
    
    // Remove entry from cache
    @CacheEvict(value = "users", key = "#id")
    public void deleteUser(Long id) {
        userRepository.deleteById(id);
    }
    
    // Clear entire cache
    @CacheEvict(value = "users", allEntries = true)
    public void clearAllUserCache() {
        System.out.println("Cache cleared for all users");
    }
}

// Redis Configuration (alternative to in-memory cache)
/*
pom.xml dependency:
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-data-redis</artifactId>
</dependency>

application.yml:
spring:
  redis:
    host: localhost
    port: 6379
    timeout: 60000ms
    database: 0
  cache:
    type: redis
    redis:
      time-to-live: 3600000  # 1 hour
*/

## 13. API Documentation with Swagger/OpenAPI

Generate interactive API documentation using SpringDoc OpenAPI (Swagger UI).

In [ ]:
// OpenAPI Configuration
package com.example.config;

import io.swagger.v3.oas.models.OpenAPI;
import io.swagger.v3.oas.models.info.Info;
import io.swagger.v3.oas.models.info.Contact;
import io.swagger.v3.oas.models.info.License;
import org.springframework.context.annotation.Bean;
import org.springframework.context.annotation.Configuration;

@Configuration
public class OpenAPIConfig {
    
    @Bean
    public OpenAPI customOpenAPI() {
        return new OpenAPI()
            .info(new Info()
                .title("Spreadsheet API")
                .version("1.0.0")
                .description("Production-ready REST API for Spreadsheet application")
                .contact(new Contact()
                    .name("API Support")
                    .email("support@example.com")
                    .url("https://example.com"))
                .license(new License()
                    .name("Apache 2.0")
                    .url("https://www.apache.org/licenses/LICENSE-2.0.html")));
    }
}

// Controller with Swagger Documentation
package com.example.controller;

import com.example.model.User;
import io.swagger.v3.oas.annotations.Operation;
import io.swagger.v3.oas.annotations.Parameter;
import io.swagger.v3.oas.annotations.media.Content;
import io.swagger.v3.oas.annotations.media.Schema;
import io.swagger.v3.oas.annotations.responses.ApiResponse;
import io.swagger.v3.oas.annotations.tags.Tag;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;

import java.util.List;

@RestController
@RequestMapping("/api/users")
@Tag(name = "User Management", description = "APIs for managing users")
public class DocumentedUserController {
    
    @GetMapping
    @Operation(summary = "Get all users", description = "Retrieve a list of all users in the system")
    @ApiResponse(responseCode = "200", description = "Successfully retrieved list of users",
            content = @Content(mediaType = "application/json", schema = @Schema(implementation = User.class)))
    public ResponseEntity<List<User>> getAllUsers() {
        return ResponseEntity.ok(List.of());
    }
    
    @GetMapping("/{id}")
    @Operation(summary = "Get user by ID", description = "Retrieve a specific user by their ID")
    @ApiResponse(responseCode = "200", description = "User found")
    @ApiResponse(responseCode = "404", description = "User not found")
    public ResponseEntity<User> getUserById(
            @Parameter(description = "User ID", required = true)
            @PathVariable Long id) {
        return ResponseEntity.ok(new User());
    }
    
    @PostMapping
    @Operation(summary = "Create new user", description = "Create a new user with provided details")
    @ApiResponse(responseCode = "201", description = "User created successfully")
    @ApiResponse(responseCode = "400", description = "Invalid input provided")
    public ResponseEntity<User> createUser(
            @io.swagger.v3.oas.annotations.parameters.RequestBody(description = "User object to create", required = true)
            @RequestBody User user) {
        return ResponseEntity.ok(user);
    }
    
    @PutMapping("/{id}")
    @Operation(summary = "Update user", description = "Update an existing user")
    public ResponseEntity<User> updateUser(
            @Parameter(description = "User ID", required = true)
            @PathVariable Long id,
            @RequestBody User user) {
        return ResponseEntity.ok(user);
    }
    
    @DeleteMapping("/{id}")
    @Operation(summary = "Delete user", description = "Delete a user by ID")
    @ApiResponse(responseCode = "200", description = "User deleted successfully")
    @ApiResponse(responseCode = "404", description = "User not found")
    public ResponseEntity<String> deleteUser(
            @Parameter(description = "User ID", required = true)
            @PathVariable Long id) {
        return ResponseEntity.ok("User deleted");
    }
}

// pom.xml dependency:
/*
<dependency>
    <groupId>org.springdoc</groupId>
    <artifactId>springdoc-openapi-starter-webmvc-ui</artifactId>
    <version>2.0.2</version>
</dependency>

Access Swagger UI at: http://localhost:8080/swagger-ui.html
Access OpenAPI JSON at: http://localhost:8080/v3/api-docs
*/

## 14. Application Monitoring with Spring Boot Actuator

Monitor application health, metrics, and performance in production environments.

In [ ]:
# Actuator Configuration (application.yml)
management:
  endpoints:
    web:
      exposure:
        include: health,info,metrics,prometheus,threaddump,heapdump
  endpoint:
    health:
      show-details: always
  metrics:
    export:
      prometheus:
        enabled: true
  health:
    livenessState:
      enabled: true
    readinessState:
      enabled: true

# Endpoints available:
# /actuator/health - Application health status
# /actuator/health/liveness - Kubernetes liveness probe
# /actuator/health/readiness - Kubernetes readiness probe
# /actuator/info - Application info
# /actuator/metrics - Available metrics
# /actuator/metrics/{metric.name} - Specific metric
# /actuator/prometheus - Prometheus format metrics
# /actuator/threaddump - Thread information
# /actuator/heapdump - Memory heap dump
# /actuator/loggers - Logger configuration
# /actuator/configprops - Configuration properties

## 15. Docker & Containerization

Deploy Spring Boot applications using Docker for consistency across environments.

In [ ]:
# Dockerfile
FROM openjdk:17-jdk-slim as builder
WORKDIR /app
COPY pom.xml .
COPY src ./src
RUN apt-get update && apt-get install -y maven
RUN mvn clean package -DskipTests

FROM openjdk:17-jdk-slim
WORKDIR /app
COPY --from=builder /app/target/*.jar app.jar

EXPOSE 8080

ENTRYPOINT ["java", "-jar", "app.jar"]

# Health check
HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
  CMD curl -f http://localhost:8080/actuator/health || exit 1

# Build: docker build -t my-app:1.0 .
# Run: docker run -p 8080:8080 my-app:1.0
# Run with MySQL: docker run -p 8080:8080 \
#   -e SPRING_DATASOURCE_URL=jdbc:mysql://mysql:3306/db \
#   -e SPRING_DATASOURCE_USERNAME=root \
#   -e SPRING_DATASOURCE_PASSWORD=password \
#   my-app:1.0

In [ ]:
# docker-compose.yml
version: '3.8'

services:
  app:
    build: .
    ports:
      - "8080:8080"
    environment:
      SPRING_DATASOURCE_URL: jdbc:mysql://mysql:3306/spreadsheet_db
      SPRING_DATASOURCE_USERNAME: root
      SPRING_DATASOURCE_PASSWORD: rootpassword
      SPRING_JPA_HIBERNATE_DDL_AUTO: update
    depends_on:
      mysql:
        condition: service_healthy
    networks:
      - app-network

  mysql:
    image: mysql:8.0
    ports:
      - "3306:3306"
    environment:
      MYSQL_ROOT_PASSWORD: rootpassword
      MYSQL_DATABASE: spreadsheet_db
    volumes:
      - mysql-data:/var/lib/mysql
    healthcheck:
      test: ["CMD", "mysqladmin", "ping", "-h", "localhost"]
      timeout: 20s
      retries: 10
    networks:
      - app-network

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    networks:
      - app-network

volumes:
  mysql-data:

networks:
  app-network:
    driver: bridge

# Usage:
# docker-compose up -d          # Start all services
# docker-compose down           # Stop all services
# docker-compose logs -f app    # View app logs
# docker-compose ps             # List running containers

## 16. Transaction Management

Implement ACID compliance using Spring's @Transactional annotation for data consistency.

In [ ]:
package com.example.service;

import com.example.model.User;
import com.example.model.Order;
import com.example.repository.UserRepository;
import com.example.repository.OrderRepository;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.stereotype.Service;
import org.springframework.transaction.annotation.Transactional;
import org.springframework.transaction.annotation.Propagation;

@Service
public class TransactionalUserService {
    
    @Autowired
    private UserRepository userRepository;
    
    @Autowired
    private OrderRepository orderRepository;
    
    // Basic transactional method - rollback on any exception
    @Transactional
    public void createUserWithOrder(User user, Order order) {
        User savedUser = userRepository.save(user);
        order.setUserId(savedUser.getId());
        orderRepository.save(order);
        // If any exception throws, both saves will be rolled back
    }
    
    // Read-only optimization
    @Transactional(readOnly = true)
    public User getUser(Long id) {
        return userRepository.findById(id).orElseThrow();
    }
    
    // Isolation levels
    @Transactional(isolation = org.springframework.transaction.annotation.Isolation.SERIALIZABLE)
    public void processPayment(Long orderId, double amount) {
        Order order = orderRepository.findById(orderId).orElseThrow();
        order.setAmount(amount);
        order.setStatus("PAID");
        orderRepository.save(order);
    }
    
    // Propagation control
    @Transactional(propagation = Propagation.REQUIRES_NEW)
    public void logTransaction(String message) {
        // Creates a new transaction independent of parent
        System.out.println("Logging: " + message);
    }
    
    // Timeout after 5 seconds
    @Transactional(timeout = 5)
    public void heavyDatabaseOperation() {
        // Transaction will rollback if not completed within 5 seconds
        userRepository.findAll();
    }
    
    // No rollback on specific exception
    @Transactional(noRollbackFor = IllegalArgumentException.class)
    public void createUser(User user) throws IllegalArgumentException {
        if (user.getName() == null) {
            throw new IllegalArgumentException("Name cannot be null");
            // This exception will NOT trigger rollback
        }
        userRepository.save(user);
    }
}

// Transactional Propagation Types:
/*
REQUIRED (default) - Use existing or create new transaction
REQUIRES_NEW - Always create new transaction, suspend existing
MANDATORY - Must execute in existing transaction, error if none
SUPPORTS - Use transaction if exists, otherwise execute without
NOT_SUPPORTED - Execute without transaction, suspend if exists
NEVER - Error if transaction exists
NESTED - Execute in nested transaction if supported by DB

Isolation Levels:
READ_UNCOMMITTED - Dirty reads allowed
READ_COMMITTED - Prevent dirty reads
REPEATABLE_READ - Prevent dirty and unrepeatable reads
SERIALIZABLE - Full isolation between transactions
*/

## 17. Integration Testing

Write comprehensive integration tests to verify component interactions and database operations.

In [ ]:
package com.example;

import com.example.model.User;
import com.example.repository.UserRepository;
import org.junit.jupiter.api.BeforeEach;
import org.junit.jupiter.api.Test;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.boot.test.autoconfigure.web.servlet.AutoConfigureMockMvc;
import org.springframework.boot.test.context.SpringBootTest;
import org.springframework.http.MediaType;
import org.springframework.test.context.ActiveProfiles;
import org.springframework.test.web.servlet.MockMvc;
import org.springframework.transaction.annotation.Transactional;

import static org.springframework.test.web.servlet.request.MockMvcRequestBuilders.*;
import static org.springframework.test.web.servlet.result.MockMvcResultMatchers.*;
import static org.hamcrest.Matchers.*;

@SpringBootTest
@AutoConfigureMockMvc
@ActiveProfiles("test")
@Transactional
public class UserIntegrationTest {
    
    @Autowired
    private MockMvc mockMvc;
    
    @Autowired
    private UserRepository userRepository;
    
    @BeforeEach
    public void setup() {
        userRepository.deleteAll();
    }
    
    @Test
    public void testCreateAndRetrieveUser() throws Exception {
        // Create
        String userJson = "{\"name\":\"John Doe\",\"email\":\"john@example.com\"}";
        mockMvc.perform(post("/api/users")
                .contentType(MediaType.APPLICATION_JSON)
                .content(userJson))
                .andExpect(status().isCreated());
        
        // Verify in database
        User user = userRepository.findByEmail("john@example.com");
        assert user != null;
        assert user.getName().equals("John Doe");
    }
    
    @Test
    public void testGetAllUsers() throws Exception {
        // Insert test data
        User user = new User("Alice", "alice@example.com");
        userRepository.save(user);
        
        // Test endpoint
        mockMvc.perform(get("/api/users")
                .contentType(MediaType.APPLICATION_JSON))
                .andExpect(status().isOk())
                .andExpect(jsonPath("$", hasSize(greaterThanOrEqualTo(1))))
                .andExpect(jsonPath("$[0].name", is("Alice")));
    }
    
    @Test
    public void testUpdateUser() throws Exception {
        // Insert initial user
        User user = new User("Bob", "bob@example.com");
        User savedUser = userRepository.save(user);
        
        // Update
        String updateJson = "{\"name\":\"Bob Updated\",\"email\":\"bob@example.com\"}";
        mockMvc.perform(put("/api/users/" + savedUser.getId())
                .contentType(MediaType.APPLICATION_JSON)
                .content(updateJson))
                .andExpect(status().isOk());
        
        // Verify update
        User updated = userRepository.findById(savedUser.getId()).orElseThrow();
        assert updated.getName().equals("Bob Updated");
    }
    
    @Test
    public void testValidationError() throws Exception {
        String invalidUserJson = "{\"name\":\"\",\"email\":\"invalid-email\"}";
        mockMvc.perform(post("/api/users")
                .contentType(MediaType.APPLICATION_JSON)
                .content(invalidUserJson))
                .andExpect(status().isBadRequest());
    }
}

// Test Configuration (application-test.yml)
/*
spring:
  datasource:
    url: jdbc:h2:mem:testdb
    driver-class-name: org.h2.Driver
  jpa:
    database-platform: org.hibernate.dialect.H2Dialect
    hibernate:
      ddl-auto: create-drop
  h2:
    console:
      enabled: true
*/

## 18. Async Processing & Scheduled Tasks

Implement non-blocking operations and scheduled tasks for better performance and scalability.

In [ ]:
// Enable Async & Scheduling Configuration
package com.example.config;

import org.springframework.context.annotation.Configuration;
import org.springframework.scheduling.annotation.EnableAsync;
import org.springframework.scheduling.annotation.EnableScheduling;
import org.springframework.context.annotation.Bean;
import org.springframework.core.task.TaskExecutor;
import org.springframework.scheduling.concurrent.ThreadPoolTaskExecutor;

@Configuration
@EnableAsync
@EnableScheduling
public class AsyncConfig {
    
    @Bean(name = "taskExecutor")
    public TaskExecutor taskExecutor() {
        ThreadPoolTaskExecutor executor = new ThreadPoolTaskExecutor();
        executor.setCorePoolSize(2);
        executor.setMaxPoolSize(5);
        executor.setQueueCapacity(100);
        executor.setThreadNamePrefix("AsyncThread-");
        executor.initialize();
        return executor;
    }
}

// Async Service Example
package com.example.service;

import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.mail.SimpleMailMessage;
import org.springframework.mail.javamail.JavaMailSender;
import org.springframework.scheduling.annotation.Async;
import org.springframework.scheduling.annotation.Scheduled;
import org.springframework.stereotype.Service;
import java.util.concurrent.CompletableFuture;

@Service
public class AsyncService {
    
    @Autowired
    private JavaMailSender mailSender;
    
    // Async method - returns immediately, executes in background
    @Async("taskExecutor")
    public void sendEmailAsync(String to, String subject, String body) {
        SimpleMailMessage message = new SimpleMailMessage();
        message.setTo(to);
        message.setSubject(subject);
        message.setText(body);
        mailSender.send(message);
        System.out.println("Email sent to: " + to);
    }
    
    // Async with return value
    @Async("taskExecutor")
    public CompletableFuture<String> processDataAsync(String data) {
        try {
            // Simulate long-running operation
            Thread.sleep(5000);
            return CompletableFuture.completedFuture("Processed: " + data);
        } catch (InterruptedException e) {
            return CompletableFuture.failedFuture(e);
        }
    }
    
    // Scheduled task - runs every 60 seconds
    @Scheduled(fixedRate = 60000)
    public void cleanExpiredTokens() {
        System.out.println("Running cleanup task...");
        // Cleanup logic here
    }
    
    // Scheduled with initial delay and fixed rate
    @Scheduled(initialDelay = 10000, fixedRate = 30000)
    public void generateDailyReport() {
        System.out.println("Generating report...");
    }
    
    // Cron expression - runs daily at 2 AM
    @Scheduled(cron = "0 0 2 * * *")
    public void runDailyBackup() {
        System.out.println("Running daily backup...");
    }
    
    // Cron expressions:
    // "0 0 * * * *" - Every hour
    // "0 0 0 * * *" - Every day at midnight
    // "0 0 0 * * MON" - Every Monday at midnight
    // "0 */5 * * * *" - Every 5 minutes
}

// Controller using Async service
package com.example.controller;

import com.example.service.AsyncService;
import org.springframework.beans.factory.annotation.Autowired;
import org.springframework.http.ResponseEntity;
import org.springframework.web.bind.annotation.*;
import java.util.concurrent.CompletableFuture;

@RestController
@RequestMapping("/api/async")
public class AsyncController {
    
    @Autowired
    private AsyncService asyncService;
    
    @PostMapping("/send-email")
    public ResponseEntity<String> sendEmail(@RequestParam String email) {
        // Returns immediately
        asyncService.sendEmailAsync(email, "Welcome", "Hello user!");
        return ResponseEntity.accepted().body("Email will be sent shortly");
    }
    
    @PostMapping("/process-data")
    public CompletableFuture<ResponseEntity<String>> processData(@RequestBody String data) {
        return asyncService.processDataAsync(data)
                .thenApply(result -> ResponseEntity.ok(result))
                .exceptionally(ex -> ResponseEntity.status(500).build());
    }
}

## 19. Production Deployment Checklist

**Infrastructure & DevOps**
- ✓ Containerize with Docker and Docker Compose
- ✓ Set up CI/CD pipelines (GitHub Actions, Jenkins, GitLab CI)
- ✓ Configure load balancing (Nginx, HAProxy)
- ✓ Implement auto-scaling policies
- ✓ Set up monitoring dashboards (Prometheus, Grafana)
- ✓ Configure log aggregation (ELK Stack, Datadog)
- ✓ Backup and disaster recovery plan

**Security**
- ✓ Implement HTTPS/TLS encryption
- ✓ Enable CORS only for trusted origins
- ✓ Use Spring Security for authentication/authorization
- ✓ Implement JWT or OAuth 2.0 for API security
- ✓ Validate and sanitize all inputs
- ✓ Use environment variables for sensitive config
- ✓ Enable CSRF protection
- ✓ Implement rate limiting
- ✓ Regular security audits and dependency updates

**Performance & Scalability**
- ✓ Implement caching (Redis, Memcached)
- ✓ Database query optimization and indexing
- ✓ Connection pooling configuration
- ✓ Implement pagination for large datasets
- ✓ Compress API responses (gzip)
- ✓ Use async processing for non-blocking operations
- ✓ Configure appropriate thread pools
- ✓ Load testing and benchmarking

**Monitoring & Observability**
- ✓ Enable Spring Boot Actuator endpoints
- ✓ Export metrics to Prometheus
- ✓ Set up health checks and probes
- ✓ Implement structured logging (SLF4J)
- ✓ Distributed tracing (Sleuth, Jaeger)
- ✓ Alert rules and notifications
- ✓ Performance metrics collection

**Data & Database**
- ✓ Database connection pooling (HikariCP)
- ✓ Implement transaction management
- ✓ Regular backups and replication
- ✓ Data validation and constraints
- ✓ Prepared statements to prevent SQL injection
- ✓ Database migrations (Flyway, Liquibase)

**Testing & Quality**
- ✓ Unit tests (JUnit 5, Mockito)
- ✓ Integration tests
- ✓ End-to-end tests
- ✓ API testing (Postman, REST Assured)
- ✓ Performance testing
- ✓ Code coverage > 80%
- ✓ Static code analysis (SonarQube)
- ✓ Dependency scanning for vulnerabilities

**Configuration Management**
- ✓ Environment-specific configurations
- ✓ Externalized configuration (application.yml)
- ✓ Feature flags for gradual rollouts
- ✓ Configuration versioning
- ✓ Secret management (AWS Secrets Manager, Vault)

**Documentation**
- ✓ API documentation (Swagger/OpenAPI)
- ✓ Architecture documentation
- ✓ Deployment procedures
- ✓ Troubleshooting guides
- ✓ Performance tuning guidelines

## 20. Production-Ready Best Practices

### Code Quality
1. **Clean Code Principles** - SOLID principles, DRY, KISS
2. **Dependency Injection** - Use constructor injection over field injection
3. **Error Handling** - Global exception handling with meaningful error messages
4. **Logging** - Structured logging with appropriate log levels
5. **Code Review** - Consistent code review process

### Database Design
1. **Normalization** - Design schemas to avoid redundancy
2. **Indexing** - Index frequently queried columns
3. **Connection Pooling** - Configure HikariCP (default in Spring Boot)
4. **Query Optimization** - Use JPA projections, pagination
5. **Migration Tools** - Use Flyway or Liquibase for schema versioning

### API Design
1. **RESTful Principles** - Proper HTTP verbs and status codes
2. **Versioning** - API versioning strategy (/api/v1/...)
3. **Rate Limiting** - Prevent abuse
4. **CORS Configuration** - Restrict to trusted domains
5. **Pagination** - Support large datasets efficiently

### Security
1. **Authentication** - JWT, OAuth 2.0, or SAML
2. **Authorization** - Role-based access control (RBAC)
3. **Encryption** - HTTPS for all endpoints
4. **Input Validation** - Server-side validation for all inputs
5. **Secrets Management** - Never hardcode credentials

### Performance
1. **Caching Strategy** - Cache frequently accessed data
2. **Async Processing** - Non-blocking operations for I/O
3. **Database Optimization** - Query tuning, connection pooling
4. **Load Balancing** - Distribute traffic across instances
5. **Monitoring** - Real-time performance metrics

### Deployment
1. **Containerization** - Docker for consistent environments
2. **CI/CD Pipeline** - Automated build, test, deploy
3. **Health Checks** - Implement readiness/liveness probes
4. **Graceful Shutdown** - Handle shutdown signals properly
5. **Configuration Management** - Environment-specific configs

### Common Production Issues & Solutions

| Issue | Solution |
|-------|----------|
| High memory usage | Analyze heap dumps, tune GC settings, implement caching |
| Slow queries | Add indexes, use query pagination, optimize JOIN operations |
| Connection pool exhaustion | Increase pool size, review connection usage patterns |
| Clustering issues | Implement distributed sessions, use Redis/Hazelcast |
| Race conditions | Use @Transactional, implement proper locking |
| Message loss | Use message queues (RabbitMQ, Kafka), implement retries |
| Cascading failures | Implement circuit breakers, timeouts, fallbacks |

## Quick Reference: Production-Ready Dependencies

**pom.xml Essential Dependencies**
```xml
<!-- Web Framework -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-web</artifactId>
</dependency>

<!-- Database -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-data-jpa</artifactId>
</dependency>
<dependency>
    <groupId>mysql</groupId>
    <artifactId>mysql-connector-java</artifactId>
</dependency>

<!-- Security -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-security</artifactId>
</dependency>
<dependency>
    <groupId>io.jsonwebtoken</groupId>
    <artifactId>jjwt</artifactId>
</dependency>

<!-- Validation -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-validation</artifactId>
</dependency>

<!-- API Documentation -->
<dependency>
    <groupId>org.springdoc</groupId>
    <artifactId>springdoc-openapi-starter-webmvc-ui</artifactId>
</dependency>

<!-- Caching & Redis -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-data-redis</artifactId>
</dependency>

<!-- Monitoring -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-actuator</artifactId>
</dependency>
<dependency>
    <groupId>io.micrometer</groupId>
    <artifactId>micrometer-registry-prometheus</artifactId>
</dependency>

<!-- Database Migrations -->
<dependency>
    <groupId>org.flywaydb</groupId>
    <artifactId>flyway-core</artifactId>
</dependency>

<!-- Testing -->
<dependency>
    <groupId>org.springframework.boot</groupId>
    <artifactId>spring-boot-starter-test</artifactId>
    <scope>test</scope>
</dependency>
```

## Learning Path Summary

1. **Foundation (Sections 1-3)** - Setup and configuration
2. **Core Features (Sections 4-8)** - REST endpoints, DI, validation, testing
3. **Production Essentials (Sections 9-18)** - Error handling, security, monitoring, deployment
4. **Advanced Topics** - Microservices, cloud deployment, advanced caching

## Resources

- [Spring Boot Official Documentation](https://spring.io/projects/spring-boot)
- [Spring Security Reference](https://spring.io/projects/spring-security)
- [Spring Data JPA Documentation](https://spring.io/projects/spring-data-jpa)
- [SpringDoc OpenAPI](https://springdoc.org/)
- [Micrometer Metrics](https://micrometer.io/)

Congratulations! You now have a comprehensive guide from fundamentals to production-ready Spring Boot applications! 🚀